In [1]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import json


# --- 1. Утилита для группировки и создания списков файлов ---

def create_grouped_file_lists(root_dir, split_ratio=0.8, seed=42, save_split_path="split_config.json"):
    """
    Возвращает два списка словарей (data items): train_list и test_list.
    Гарантирует, что аугментации одного и того же изображения попадают строго
    либо в трейн, либо в тест.
    """
    # Словарь: {base_id: [список полных путей к файлам этой группы во всех distance папках]}
    # Но у нас структура сложнее: одна "сцена" (частицы) повторяется в разных distance папках.
    # Нам нужно, чтобы сцена "X" во ВСЕХ расстояниях и ВСЕХ аугментациях попала либо в трейн, либо в тест.
    
    # 1. Собираем все доступные файлы и группируем их по ID сцены
    # ID сцены - это имя файла без суффикса аугментации.
    # Пример: '0_1_0_0_180.png' -> base_id = '0_1_0_0'
    # Пример: '0_1_0_0.png'     -> base_id = '0_1_0_0'
    
    groups = defaultdict(list) # {base_id: [{'input': path, 'target': path, 'dist': val}, ...]}
    all_distances = []

    if not os.path.exists(root_dir):
        raise FileNotFoundError(f"Папка {root_dir} не найдена")

    print("Сканирование файлов и группировка по ID...")
    
    for folder_name in sorted(os.listdir(root_dir)):
        folder_path = os.path.join(root_dir, folder_name)
        if os.path.isdir(folder_path) and "distance" in folder_name:
            try:
                dist_val = float(folder_name.split(' ')[-1])
                all_distances.append(dist_val)
                
                input_dir = os.path.join(folder_path, 'input')
                target_dir = os.path.join(folder_path, 'target')
                
                if os.path.exists(input_dir) and os.path.exists(target_dir):
                    fnames = sorted(os.listdir(input_dir))
                    for fname in fnames:
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            # --- ЛОГИКА ИЗВЛЕЧЕНИЯ BASE_ID ---
                            # Предполагаем формат: ID_части_аугментация.png
                            # Если аугментации (углы) идут в конце через '_', то отрезаем последнее число
                            # А если просто имя файла уникально для сцены, то берем имя без расширения.
                            
                            name_no_ext = os.path.splitext(fname)[0]
                            
                            # Эвристика: если есть аугментации типа _90, _180, то они обычно в конце.
                            # Твой пример: 0_1_0_0_180.png -> base = 0_1_0_0
                            #              0_1_0_0.png     -> base = 0_1_0_0
                            
                            parts = name_no_ext.split('_')
                            # Проверка: является ли последняя часть числом (угол)?
                            # Если да (и это аугментация), отбрасываем её.
                            # ВНИМАНИЕ: Это зависит от именования. 
                            # Если 0_1_0_0 это ID, а 180 это угол, то:
                            
                            # Простейший вариант: считаем "группой" всё, что имеет общий префикс до последнего подчеркивания,
                            # ЕСЛИ последнее - это угол. Если нет - то всё имя ID.
                            # Чтобы не гадать, давай считать ID всё имя файла БЕЗ расширения.
                            # НО ты сказал: "исходная 0_1_0_0 и аугментированная 0_1_0_0_180".
                            # Значит 0_1_0_0_180 содержит в себе ID 0_1_0_0.
                            
                            if len(parts) > 4: # Если частей много, скорее всего есть суффикс
                                # Попытаемся определить, является ли хвост углом
                                last_part = parts[-1]
                                if last_part.replace('+','').replace('-','').isdigit():
                                    # Это аугментация, ID - это всё кроме хвоста
                                    base_id = "_".join(parts[:-1])
                                else:
                                    # Это не аугментация, это просто длинное имя
                                    base_id = name_no_ext
                            else:
                                base_id = name_no_ext

                            # Собираем пару
                            inp_path = os.path.join(input_dir, fname)
                            tar_path = os.path.join(target_dir, fname)
                            
                            if os.path.exists(tar_path):
                                groups[base_id].append({
                                    'input': inp_path,
                                    'target': tar_path,
                                    'dist': dist_val
                                })
            except Exception as e:
                print(f"Error processing {folder_name}: {e}")
                continue

    # 2. Делим ГРУППЫ (ID) на train и test
    unique_ids = list(groups.keys())
    random.seed(seed)
    random.shuffle(unique_ids)
    
    split_idx = int(len(unique_ids) * split_ratio)
    train_ids = unique_ids[:split_idx]
    test_ids = unique_ids[split_idx:]

    #Сохраняем ID тестовой выборки в файл
    split_data = {
        "seed": seed,
        "split_ratio": split_ratio,
        "train_ids": train_ids,
        "test_ids": test_ids,
        "max_dist": max(all_distances) if all_distances else 1.0
    }
    
    with open(save_split_path, 'w') as f:
        json.dump(split_data, f, indent=4)
    print(f"Конфигурация сплита сохранена в {save_split_path}")
    
    train_data = []
    test_data = []
    
    # Разворачиваем группы обратно в плоский список файлов
    for uid in train_ids:
        train_data.extend(groups[uid])
        
    for uid in test_ids:
        test_data.extend(groups[uid])
        
    max_dist = max(all_distances) if all_distances else 1.0
    
    print(f"Total groups (unique scenes): {len(unique_ids)}")
    print(f"Train groups: {len(train_ids)} -> {len(train_data)} images")
    print(f"Test groups:  {len(test_ids)}  -> {len(test_data)} images")
    
    return train_data, test_data, split_data["max_dist"]

In [2]:
class PolymerListDataset(Dataset):
    def __init__(self, data_list, max_dist, size=(512, 512)):
        self.data = data_list
        self.max_dist = max_dist
        self.size = size
        self.transform = transforms.Compose([
            transforms.Resize(size),
            transforms.ToTensor(), # [0,1]
            transforms.Normalize((0.5,), (0.5,)) # [-1,1]
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        input_img = Image.open(item['input']).convert('L')
        target_img = Image.open(item['target']).convert('L')
        
        input_tensor = self.transform(input_img)
        target_tensor = self.transform(target_img)
        
        # Нормализация дистанции [0, 1]
        norm_dist = item['dist'] / self.max_dist
        dist_tensor = torch.tensor([norm_dist], dtype=torch.float32)
        
        return input_tensor, target_tensor, dist_tensor

In [3]:
# 2. АРХИТЕКТУРА МОДЕЛЕЙ (CONDITIONAL)
# ==========================================

class ConditionalUNet(nn.Module):
    def __init__(self):
        super(ConditionalUNet, self).__init__()
        # Вход: 2 канала (Картинка + Дистанция)
        self.conv_in = nn.Conv2d(2, 64, kernel_size=4, stride=2, padding=1, bias=False)
        
        # Encoder
        self.down1 = nn.Sequential(nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128))
        self.down2 = self._make_down_block(128, 256)
        self.down3 = self._make_down_block(256, 512)
        self.down4 = self._make_down_block(512, 512)
        self.down5 = self._make_down_block(512, 512)
        self.down6 = self._make_down_block(512, 512)

        # Middle
        self.middle = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, 4, 2, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512)
        )

        # Decoder
        self.up1 = self._make_up_block(1024, 512, dropout=True)
        self.up2 = self._make_up_block(1024, 512, dropout=True)
        self.up3 = self._make_up_block(1024, 512, dropout=True)
        self.up4 = self._make_up_block(1024, 256)
        self.up5 = self._make_up_block(512, 128)
        self.up6 = self._make_up_block(256, 64)

        self.outermost = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh()
        )

    def _make_down_block(self, in_channels, out_channels):
        return nn.Sequential(nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False), nn.BatchNorm2d(out_channels))

    def _make_up_block(self, in_channels, out_channels, dropout=False):
        layers = [nn.ReLU(inplace=True), nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False), nn.BatchNorm2d(out_channels)]
        if dropout: layers.append(nn.Dropout(0.5))
        return nn.Sequential(*layers)

    def forward(self, x, distance):
        # Spatial Replication: Растягиваем число расстояния до размера картинки
        B, _, H, W = x.shape
        dist_map = distance.view(B, 1, 1, 1).expand(B, 1, H, W)
        
        # Конкатенация
        x_in = torch.cat([x, dist_map], dim=1) # Теперь 2 канала
        
        x0 = self.conv_in(x_in)
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        x3 = self.down3(x2)
        x4 = self.down4(x3)
        x5 = self.down5(x4)
        x6 = self.down6(x5)
        
        x_mid = self.middle(x6)
        
        x = self.up1(torch.cat([x_mid, x6], dim=1))
        x = self.up2(torch.cat([x, x5], dim=1))
        x = self.up3(torch.cat([x, x4], dim=1))
        x = self.up4(torch.cat([x, x3], dim=1))
        x = self.up5(torch.cat([x, x2], dim=1))
        x = self.up6(torch.cat([x, x1], dim=1))
        
        return self.outermost(torch.cat([x, x0], dim=1))

In [4]:
class ConditionalPatchGAN(nn.Module):
    def __init__(self):
        super(ConditionalPatchGAN, self).__init__()
        # Вход: 3 канала (Input + Target/Fake + Distance)
        layers = [
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            self._block(64, 128),
            self._block(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 1)
        ]
        self.net = nn.Sequential(*layers)

    def _block(self, in_ch, out_ch):
        return nn.Sequential(nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True))

    def forward(self, img_A, img_B, distance):
        B, _, H, W = img_A.shape
        dist_map = distance.view(B, 1, 1, 1).expand(B, 1, H, W)
        x = torch.cat([img_A, img_B, dist_map], dim=1)
        return self.net(x)

In [5]:
# 3.ВИЗУАЛИЗАЦИЯ

class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):
        self.n_epochs = n_epochs; self.offset = offset; self.decay_start_epoch = decay_start_epoch
    def step(self, epoch):
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / (self.n_epochs - self.decay_start_epoch)

def visualize_results(inputs, targets, outputs, dists, max_dist, epoch, save_dir="training_results"):
    os.makedirs(save_dir, exist_ok=True)
    inputs = (inputs.cpu() + 1) / 2
    targets = (targets.cpu() + 1) / 2
    outputs = (outputs.cpu() + 1) / 2
    
    # Берем 3 примера
    n_vis = min(3, inputs.size(0))
    fig, axes = plt.subplots(n_vis, 3, figsize=(12, 4 * n_vis))
    if n_vis == 1: axes = axes.reshape(1, -1)
    
    fig.suptitle(f"Epoch {epoch+1}", fontsize=16)
    
    for i in range(n_vis):
        real_d = dists[i].item() * max_dist
        
        axes[i, 0].imshow(inputs[i, 0], cmap='gray')
        axes[i, 0].set_title(f"Input (Dist={real_d:.2f})")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(targets[i, 0], cmap='gray')
        axes[i, 1].set_title("Target")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(outputs[i, 0], cmap='gray')
        axes[i, 2].set_title("Generated")
        axes[i, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f"{save_dir}/epoch_{epoch+1}.png")
    plt.close()

In [6]:
def plot_training_history(history, save_dir="training_results"):
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history['G_loss']) + 1)
    
    plt.figure(figsize=(18, 5))
    
    # 1. График Лоссов (G vs D)
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['G_loss'], label='Generator Loss', color='#2ca02c')
    plt.plot(epochs, history['D_loss'], label='Discriminator Loss', color='#d62728')
    plt.title('Generator & Discriminator Losses')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 2. График вероятностей D(x) и D(G(z))
    # Это показывает "битву": D(x) должно быть высоко, D(G(z)) должно расти, если G учится обманывать
    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['D_x'], label='D(x) - Real Prob', color='#1f77b4')
    plt.plot(epochs, history['D_G_z'], label='D(G(z)) - Fake Prob', color='#ff7f0e')
    plt.title('Discriminator Probabilities')
    plt.xlabel('Epoch')
    plt.ylabel('Probability (0 to 1)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.05)
    
    # 3. График L1 Loss (насколько картинка похожа пиксельно)
    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['G_L1'], label='L1 Loss', color='#9467bd')
    plt.title('L1 Pixel-wise Loss')
    plt.xlabel('Epoch')
    plt.ylabel('L1 Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'final_training_curves.png'))
    plt.show()


In [7]:
import os

# === УКАЖИ СВОЙ ПУТЬ ===
ROOT_DIR = '/kaggle/input/polymermodelingdataset/Распределение поля2' 
# =======================

print(f"--- ДИАГНОСТИКА ДАТАСЕТА: {ROOT_DIR} ---")

if not os.path.exists(ROOT_DIR):
    print(f" ОШИБКА: Папка {ROOT_DIR} вообще не существует!")
else:
    print(f" Папка найдена. Содержимое корня:")
    folders = sorted(os.listdir(ROOT_DIR))
    print(folders[:10]) # Показать первые 10 папок

    found_files = 0
    
    for folder_name in folders:
        folder_path = os.path.join(ROOT_DIR, folder_name)
        if not os.path.isdir(folder_path):
            continue
            
        # 1. Проверка имени папки (distance ...)
        if "distance" not in folder_name:
            print(f" Пропуск папки '{folder_name}': нет слова 'distance'")
            continue
            
        # 2. Попытка извлечь число
        try:
            # Универсальный парсер: заменяет и пробелы и подчеркивания
            clean_name = folder_name.replace('distance', '').replace('_', ' ').strip()
            dist_val = float(clean_name)
            print(f" Папка '{folder_name}' -> Дистанция распознана: {dist_val}")
        except ValueError:
            print(f"ОШИБКА в '{folder_name}': не удалось превратить '{clean_name}' в число")
            continue

        # 3. Проверка input / target
        input_dir = os.path.join(folder_path, 'input')
        target_dir = os.path.join(folder_path, 'target')
        
        if not os.path.exists(input_dir):
            print(f"  Нет папки input в {folder_name}")
            continue
        if not os.path.exists(target_dir):
            print(f"  Нет папки target в {folder_name}")
            continue
            
        # 4. Проверка файлов внутри
        in_files = os.listdir(input_dir)
        if len(in_files) == 0:
            print(f"  Папка input пустая!")
        else:
            # Проверим первый файл
            first_file = in_files[0]
            target_path = os.path.join(target_dir, first_file)
            if os.path.exists(target_path):
                print(f"  Найдена пара: {first_file}")
                found_files += 1
            else:
                print(f"  Не найдена пара для {first_file} (проверь имена в target!)")

    print("-" * 30)
    if found_files == 0:
        print("ИТОГ: Файлы не найдены. DataLoader упадет.")
    else:
        print(f"ИТОГ: Всё ок, найдено примеров: {found_files} (это не полное число, а число папок с хотя бы 1 парой).")


--- ДИАГНОСТИКА ДАТАСЕТА: /kaggle/input/polymermodelingdataset/Распределение поля2 ---
 Папка найдена. Содержимое корня:
['distance 0.032', 'distance 0.071', 'distance 0.14', 'distance 0.31', 'distance 0.45']
 Папка 'distance 0.032' -> Дистанция распознана: 0.032
  Найдена пара: 641.png
 Папка 'distance 0.071' -> Дистанция распознана: 0.071
  Найдена пара: 641.png
 Папка 'distance 0.14' -> Дистанция распознана: 0.14
  Найдена пара: 641.png
 Папка 'distance 0.31' -> Дистанция распознана: 0.31
  Найдена пара: 641.png
 Папка 'distance 0.45' -> Дистанция распознана: 0.45
  Найдена пара: 641.png
------------------------------
ИТОГ: Всё ок, найдено примеров: 5 (это не полное число, а число папок с хотя бы 1 парой).


In [ ]:
def train():
    
    ROOT_DIR = '/kaggle/input/polymermodelingdataset/Распределение поля2' # путь
    N_EPOCHS = 100
    DECAY_EPOCH = 50
    BATCH_SIZE = 4
    LR = 0.0002
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # 1. Подготовка данных
    train_list, test_list, max_dist = create_grouped_file_lists(ROOT_DIR)

    # --- БЛОК ПРОВЕРКИ НА УТЕЧКИ (DATA LEAKAGE)
    print("Проверка на Data Leakage")
    
    def get_scene_id(path):
        # Функция для извлечения "чистого" ID из пути к файлу
        # Повторяет логику из create_grouped_file_lists
        fname = os.path.basename(path)
        name_no_ext = os.path.splitext(fname)[0]
        parts = name_no_ext.split('_')
        # Если есть суффикс аугментации (число в конце), отрезаем его
        if len(parts) > 4 and parts[-1].replace('+','').replace('-','').isdigit():
            return "_".join(parts[:-1])
        return name_no_ext

    # Собираем множества ID для трейна и теста
    train_ids_set = set(get_scene_id(item['input']) for item in train_list)
    test_ids_set = set(get_scene_id(item['input']) for item in test_list)
    
    # Ищем пересечение
    intersection = train_ids_set.intersection(test_ids_set)
    
    if len(intersection) > 0:
        print(f"\n!!! ОШИБКА: НАЙДЕНА УТЕЧКА ДАННЫХ")
        print(f"Количество пересекающихся ID: {len(intersection)}")
        print(f"Примеры пересечений: {list(intersection)[:5]}")
        raise RuntimeError("Остановка: Train и Test множества пересекаются")
    else:
        print(" Проверка пройдена: Пересечений между Train и Test не найдено")
        print(f"Уникальных сцен в Train: {len(train_ids_set)}")
        print(f"Уникальных сцен в Test:  {len(test_ids_set)}")
    # ----------------------------------------------------
    
    train_ds = PolymerListDataset(train_list, max_dist)
    test_ds = PolymerListDataset(test_list, max_dist)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)
    
    # 2. Модели
    netG = ConditionalUNet().to(device)
    netD = ConditionalPatchGAN().to(device)
    
    opt_G = torch.optim.Adam(netG.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(netD.parameters(), lr=LR, betas=(0.5, 0.999))
    
    sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda=LambdaLR(N_EPOCHS, 0, DECAY_EPOCH).step)
    sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda=LambdaLR(N_EPOCHS, 0, DECAY_EPOCH).step)
    
    criterion_GAN = nn.BCEWithLogitsLoss()
    criterion_L1 = nn.L1Loss()
    
    # Фиксированный батч для визуализации
    fixed_in, fixed_tgt, fixed_dist = next(iter(test_loader))
    fixed_in, fixed_tgt, fixed_dist = fixed_in.to(device), fixed_tgt.to(device), fixed_dist.to(device)
    
    # --- ИСТОРИЯ ДЛЯ ГРАФИКОВ ---
    history = {
        'G_loss': [], 'D_loss': [],
        'D_x': [], 'D_G_z': [],
        'G_L1': []
    }
    
    print("Start Training...")
    
    for epoch in range(N_EPOCHS):
        netG.train()
        netD.train()
        
        # Аккумуляторы для среднего значения за эпоху
        epoch_G_loss = 0.0
        epoch_D_loss = 0.0
        epoch_D_x = 0.0
        epoch_D_G_z = 0.0
        epoch_L1 = 0.0
        
        loop = tqdm(train_loader, desc=f"Ep {epoch+1}/{N_EPOCHS}")
        
        for i, (imgs, targets, dists) in enumerate(loop):
            imgs, targets, dists = imgs.to(device), targets.to(device), dists.to(device)
            
            # ---------------------
            #  Train Discriminator
            # ---------------------
            opt_D.zero_grad()
            
            # 1. Real
            pred_real = netD(imgs, targets, dists)
            loss_D_real = criterion_GAN(pred_real, torch.ones_like(pred_real))
            
            # Метрика D(x): берем sigmoid, так как выход логиты, и усредняем
            d_x = torch.sigmoid(pred_real).mean().item()
            
            # 2. Fake
            fake_imgs = netG(imgs, dists)
            pred_fake = netD(imgs, fake_imgs.detach(), dists)
            loss_D_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))
            
            # Метрика D(G(z)) первая (когда D смотрит на фейк)
            d_g_z1 = torch.sigmoid(pred_fake).mean().item()
            
            loss_D = (loss_D_real + loss_D_fake) * 0.5
            loss_D.backward()
            opt_D.step()
            
            # -----------------
            #  Train Generator
            # -----------------
            opt_G.zero_grad()
            
            # G хочет обмануть D, поэтому target = ones
            pred_fake_G = netD(imgs, fake_imgs, dists)
            loss_G_GAN = criterion_GAN(pred_fake_G, torch.ones_like(pred_fake_G))
            
            # Метрика D(G(z)) вторая (после обновления D, как G смог обмануть)
            # Обычно для графика берем d_g_z1 (насколько D распознал фейк), но можно и эту.
            # Будем записывать d_g_z1 как "уверенность D в фейке" (должна быть низкой, но G тянет её вверх)
            
            loss_G_L1 = criterion_L1(fake_imgs, targets) * 100
            loss_G = loss_G_GAN + loss_G_L1
            loss_G.backward()
            opt_G.step()
            
            # --- СБОР СТАТИСТИКИ ---
            epoch_G_loss += loss_G.item()
            epoch_D_loss += loss_D.item()
            epoch_D_x += d_x
            epoch_D_G_z += d_g_z1
            epoch_L1 += loss_G_L1.item()
            
            # Обновляем прогресс-бар (показываем текущие мгновенные значения)
            loop.set_postfix(
                D_loss=f"{loss_D.item():.3f}", 
                G_loss=f"{loss_G.item():.3f}",
                D_x=f"{d_x:.2f}",       # Уверенность в реальном (идеал -> 1.0)
                D_G_z=f"{d_g_z1:.2f}"   # Уверенность в фейке (идеал D->0.0, G->1.0)
            )
            
        # --- КОНЕЦ ЭПОХИ ---
        sched_G.step()
        sched_D.step()
        
        # Усредняем метрики за эпоху
        avg_G_loss = epoch_G_loss / len(train_loader)
        avg_D_loss = epoch_D_loss / len(train_loader)
        avg_D_x = epoch_D_x / len(train_loader)
        avg_D_G_z = epoch_D_G_z / len(train_loader)
        avg_L1 = epoch_L1 / len(train_loader)
        
        # Сохраняем в историю
        history['G_loss'].append(avg_G_loss)
        history['D_loss'].append(avg_D_loss)
        history['D_x'].append(avg_D_x)
        history['D_G_z'].append(avg_D_G_z)
        history['G_L1'].append(avg_L1)
        
        # Валидация и сохранение картинок
        netG.eval()
        with torch.no_grad():
            val_fake = netG(fixed_in, fixed_dist)
            visualize_results(fixed_in, fixed_tgt, val_fake, fixed_dist, max_dist, epoch)
            
        if (epoch+1) % 10 == 0:
            torch.save(netG.state_dict(), f"Gen_ep{epoch+1}.pth")

    # --- ПОСЛЕ ВСЕГО ОБУЧЕНИЯ ---
    print("Training Finished. Plotting graphs...")
    plot_training_history(history)
    # Также сохраним саму модель
    torch.save(netG.state_dict(), "generator_final.pth")

if __name__ == "__main__":
    train()

In [25]:
import imageio
import cv2

# НАСТРОЙКИ
MODEL_PATH = '/kaggle/input/generator-modeling/generator_final (1).pth'          # Путь к весам
ROOT_DIR = '/kaggle/input/pres-data/data for pres' # Путь к датасету
SPLIT_CONFIG = '/kaggle/input/pres-data/data for pres/split_config.json'    # Файл с конфигом сплита (из train.py)
OUTPUT_ROOT = 'data for pres'
# Если в конфиге нет max_dist, используем это (но лучше из конфига)
FALLBACK_MAX_DIST = 0.45       
TARGET_DISTANCES = [0.014, 0.032, 0.071, 0.14, 0.31, 0.45]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CMAP_NAME = 'jet'


# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def create_overlay_image(generated_list, original_img_np):
    """
    Создает наложение с перекрытием слоев полимера.
    Input остается белым на переднем плане.
    Слои идут от красного (первый) к синему (последний).
    """
    H, W = original_img_np.shape
    
    # База: оригинальная картинка (Input) - останется видна везде
    base = (original_img_np * 255).astype(np.uint8)
    result = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR).astype(np.float32)
    
    # Используем reversed colormap (от красного к синему: 'jet_r' или просто инвертируем индекс)
    cmap = plt.get_cmap(CMAP_NAME)
    colors = []
    for i in range(len(generated_list)):
        # Инвертируем индекс: первый слой (i=0) -> color_idx=1.0 (красный в jet)
        #                      последний слой (i=n) -> color_idx=0.0 (синий в jet)
        color_idx = 1.0 - (i / (len(generated_list) - 1) if len(generated_list) > 1 else 0)
        color_rgba = cmap(color_idx)
        # RGB -> BGR для OpenCV
        color_bgr = np.array([color_rgba[2], color_rgba[1], color_rgba[0]]) * 255
        colors.append(color_bgr)
    
    # Накладываем слои ОТ ПОСЛЕДНЕГО К ПЕРВОМУ
    for i in range(len(generated_list) - 1, -1, -1):
        img_layer = generated_list[i]
        
        # Бинаризуем маску
        mask_binary = (img_layer > 0.5).astype(np.float32)
        
        # Маска 3-канальная
        mask_3c = np.stack([mask_binary, mask_binary, mask_binary], axis=2)  # [H, W, 3]
        
        # Где маска = 1, ставим цвет этого слоя
        for c in range(3):
            result[:, :, c] = np.where(mask_3c[:, :, c] > 0.5, colors[i][c], result[:, :, c])
    
    # Наложить Input поверх (белый слой, где частица)
    # Input уже находится в result, и он остается видным, так как мы не переписываем его
    # Но если нужно явно усилить белизну Input, можно сделать так:
    input_mask = (original_img_np > 0.5).astype(np.float32)
    input_mask_3c = np.stack([input_mask, input_mask, input_mask], axis=2)
    for c in range(3):
        result[:, :, c] = np.where(input_mask_3c[:, :, c] > 0.5, 255, result[:, :, c])
    
    return result.astype(np.uint8)




def create_gif(save_path, generated_list, original_img_np, distances):
    frames = []
    original_uint8 = (original_img_np * 255).astype(np.uint8)
    for i, gen_img in enumerate(generated_list):
        gen_uint8 = (gen_img * 255).astype(np.uint8)
        H, W = original_uint8.shape
        canvas = np.zeros((H, W*2), dtype=np.uint8)
        canvas[:, :W] = original_uint8
        canvas[:, W:] = gen_uint8
        canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
        text = f"Dist: {distances[i]:.2f}"
        cv2.putText(canvas_rgb, text, (W + 20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        frames.append(canvas_rgb)
    imageio.mimsave(save_path, frames, fps=2, loop=0)

def get_test_files_from_config(root_dir, config_path):
    """Загружает список файлов, которые были в Test при обучении"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Файл {config_path} не найден! Сначала запустите train.py.")
        
    with open(config_path, 'r') as f:
        data = json.load(f)
        
    test_ids = set(data['test_ids'])
    max_dist = data.get('max_dist', FALLBACK_MAX_DIST)
    test_files = []
    
    print(f"Загружено {len(test_ids)} тестовых ID из конфига.")
    
    for folder_name in sorted(os.listdir(root_dir)):
        folder_path = os.path.join(root_dir, folder_name)
        if os.path.isdir(folder_path) and "distance" in folder_name:
            input_dir = os.path.join(folder_path, 'Input')
            if os.path.exists(input_dir):
                for fname in sorted(os.listdir(input_dir)):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                        name_no_ext = os.path.splitext(fname)[0]
                        parts = name_no_ext.split('_')
                        if len(parts) > 4 and parts[-1].replace('+','').replace('-','').isdigit():
                            base_id = "_".join(parts[:-1])
                        else:
                            base_id = name_no_ext
                        
                        if base_id in test_ids:
                            test_files.append(os.path.join(input_dir, fname))
                            
    print(f"Найдено {len(test_files)} файлов (оригиналы + аугментации).")
    return test_files, max_dist

# ЗАПУСК ИНФЕРЕНСА
def run_inference():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    
    # 1. Загрузка модели
    generator = ConditionalUNet().to(DEVICE)
    generator.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    generator.eval()
    print(f"Модель загружена: {MODEL_PATH}")

    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 2. Получаем файлы ИЗ КОНФИГА
    test_files, max_dist_val = get_test_files_from_config(ROOT_DIR, SPLIT_CONFIG)
    print(f"Используем MAX_DIST из конфига: {max_dist_val}")

    # 3. Проход по файлам
    for img_path in tqdm(test_files, desc="Processing"):
        fname = os.path.basename(img_path)
        img_name = os.path.splitext(fname)[0]
        
        save_dir = os.path.join(OUTPUT_ROOT, img_name)
        os.makedirs(save_dir, exist_ok=True)
        
        # Загрузка картинки
        raw_pil = Image.open(img_path).convert('L')
        raw_tensor = transform(raw_pil).unsqueeze(0).to(DEVICE)
        
        results_list = []
        
        with torch.no_grad():
            for dist_val in TARGET_DISTANCES:
                norm_dist = dist_val / max_dist_val # Используем правильный max_dist
                dist_t = torch.tensor([[norm_dist]], dtype=torch.float32).to(DEVICE)
                
                out = generator(raw_tensor, dist_t)
                out_np = (out.squeeze().cpu().numpy() + 1) / 2
                results_list.append(out_np)
                
                # Сохранение отдельного кадра: name_dist.png
                v_name = f"{img_name}_{dist_val:.2f}.png"
                Image.fromarray((out_np * 255).astype(np.uint8)).save(os.path.join(save_dir, v_name))

        # Сохранение оригинала: name.png
        raw_resized = raw_pil.resize((512, 512))
        raw_resized.save(os.path.join(save_dir, f"{img_name}.png"))
        raw_np = np.array(raw_resized) / 255.0
        
        # Сохранение Overlay и GIF
        cv2.imwrite(os.path.join(save_dir, f"{img_name}_overlay.png"), create_overlay_image(results_list, raw_np))
        create_gif(os.path.join(save_dir, f"{img_name}_animation.gif"), results_list, raw_np, TARGET_DISTANCES)

    print("Инференс на тестовой выборке завершен.")

if __name__ == "__main__":
    run_inference()


Модель загружена: /kaggle/input/generator-modeling/generator_final (1).pth
Загружено 7 тестовых ID из конфига.
Найдено 42 файлов (оригиналы + аугментации).
Используем MAX_DIST из конфига: 0.45


Processing: 100%|██████████| 42/42 [00:17<00:00,  2.46it/s]

Инференс на тестовой выборке завершен.


In [26]:
import shutil
shutil.make_archive('data for pres', 'zip', '/kaggle/working/data for pres')

'/kaggle/working/data for pres.zip'

In [29]:
import os
import numpy as np
from PIL import Image
from skimage.transform import resize

def load_mask(path):
    img = Image.open(path).convert('L')
    mask = np.array(img) > 128  # threshold to binary
    return mask.astype(np.uint8)

gt_dir = '/kaggle/input/pres-data/data for pres'
pred_dir = '/kaggle/working/data for pres'

distances = ['0.014', '0.032', '0.071', '0.14', '0.31', '0.45']
images = ['1', '2']
pred_names = {'1': ['1_0.01.png', '1_0.03.png', '1_0.07.png', '1_0.14.png', '1_0.31.png', '1_0.45.png'],
              '2': ['2_0.01.png', '2_0.03.png', '2_0.07.png', '2_0.14.png', '2_0.31.png', '2_0.45.png']}

iou_scores = {}
existing_files = []

# List existing files for debugging
print("GT directories:")
for dist in distances:
    gt_dist_dir = os.path.join(gt_dir, f'distance {dist}', 'target')
    if os.path.exists(gt_dist_dir):
        files = os.listdir(gt_dist_dir)
        print(f"{gt_dist_dir}: {files}")
    else:
        print(f"GT dir not found: {gt_dist_dir}")

print("\nPrediction directories:")
for img in images:
    pred_img_dir = os.path.join(pred_dir, img)
    if os.path.exists(pred_img_dir):
        files = os.listdir(pred_img_dir)
        print(f"{pred_img_dir}: {files}")
    else:
        print(f"Pred dir not found: {pred_img_dir}")

# Compute IoUs
for img in images:
    iou_per_dist = []
    has_gt = False
    for i, dist in enumerate(distances):
        gt_path = os.path.join(gt_dir, f'distance {dist}', 'target', f'{img}.png')
        pred_filename = pred_names[img][i]
        pred_path = os.path.join(pred_dir, img, pred_filename)
        
        if os.path.exists(gt_path) and os.path.exists(pred_path):
            has_gt = True
            gt_mask = load_mask(gt_path)
            pred_mask = load_mask(pred_path)
            
            # Resize pred to gt if shapes differ
            if pred_mask.shape != gt_mask.shape:
                pred_mask = resize(pred_mask.astype(float), gt_mask.shape, anti_aliasing=True) > 0.5
                pred_mask = pred_mask.astype(np.uint8)
            
            intersection = np.logical_and(gt_mask, pred_mask).sum()
            union = np.logical_or(gt_mask, pred_mask).sum()
            iou = 1.0 if union == 0 else intersection / union
            iou_per_dist.append((dist, iou))
            print(f"IoU image {img}, dist {dist}: {iou:.4f} (gt_shape={gt_mask.shape}, pred_shape={pred_mask.shape})")
        else:
            print(f"Missing files for img {img}, dist {dist}: GT={gt_path}, Pred={pred_path}")
    
    if has_gt:
        iou_scores[img] = iou_per_dist

# Average per distance
dist_ious = {dist: [] for dist in distances}
for img, ious in iou_scores.items():
    for dist, iou in ious:
        dist_ious[dist].append(iou)

avg_dist_ious = {dist: np.mean(ious) for dist, ious in dist_ious.items() if ious}
overall_avg = np.mean(list(avg_dist_ious.values())) if avg_dist_ious else np.nan

print("\nAverage IoU per distance:")
for dist, avg in avg_dist_ious.items():
    print(f"Distance {dist}: {avg:.4f}")

print(f"\nOverall average IoU: {overall_avg:.4f}")

# CSV output for table
print("\nIoU Table (CSV format):")
print("Image,Distance,IoU")
for img, ious in iou_scores.items():
    for dist, iou in ious:
        print(f"{img},{dist},{iou:.4f}")
if avg_dist_ious:
    print("Average,Distance,Avg_IoU")
    for dist, avg in avg_dist_ious.items():
        print(f"Average,{dist},{avg:.4f}")
print(f"Overall,,{overall_avg:.4f}")

GT directories:
/kaggle/input/pres-data/data for pres/distance 0.014/target: ['1.png', '2.png']
/kaggle/input/pres-data/data for pres/distance 0.032/target: ['1.png', '2.png']
/kaggle/input/pres-data/data for pres/distance 0.071/target: ['1.png', '2.png']
/kaggle/input/pres-data/data for pres/distance 0.14/target: ['1.png', '2.png']
/kaggle/input/pres-data/data for pres/distance 0.31/target: ['1.png', '2.png']
/kaggle/input/pres-data/data for pres/distance 0.45/target: ['1.png', '2.png']

Prediction directories:
/kaggle/working/data for pres/1: ['1.png', '1_0.31.png', '1_0.03.png', '1_overlay.png', '1_0.14.png', '1_animation.gif', '1_0.01.png', '1_0.07.png', '1_0.45.png']
/kaggle/working/data for pres/2: ['2_0.45.png', '2.png', '2_0.14.png', '2_animation.gif', '2_0.03.png', '2_overlay.png', '2_0.01.png', '2_0.31.png', '2_0.07.png']
IoU image 1, dist 0.014: 0.8546 (gt_shape=(1080, 1080), pred_shape=(1080, 1080))
IoU image 1, dist 0.032: 0.9607 (gt_shape=(1080, 1080), pred_shape=(1080, 1

In [10]:
import imageio

MODEL_PATH = 'Gen_ep100.pth'         # Путь к сохраненным весам
INPUT_FOLDER = 'inference_input'     # Папка с исходными изображениями частиц
OUTPUT_ROOT = 'inference_output'     # Папка для сохранения результатов
MAX_DIST = 31.92                     # Максимальная дистанция (как при обучении)
TARGET_DISTANCES = [0.0, 2.5, 5.0, 7.5, 10.0, 15.0, 20.0, 25.0, 31.92] # Шаги генерации
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CMAP_NAME = 'jet'                    # Цветовая карта для overlay (jet, plasma, viridis)


def create_overlay_image(generated_list, original_img_np):
    """
    Создает изображение с цветными контурами слоев.
    """
    H, W = original_img_np.shape
    base = (original_img_np * 255).astype(np.uint8)
    base_color = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
    
    cmap = plt.get_cmap(CMAP_NAME)
    
    for i, img_layer in enumerate(generated_list):
        mask_uint8 = (img_layer * 255).astype(np.uint8)
        
        # Цвет зависит от индекса (градиент)
        color_idx = i / (len(generated_list) - 1) if len(generated_list) > 1 else 0
        color_rgba = cmap(color_idx)
        color_bgr = np.array([color_rgba[2], color_rgba[1], color_rgba[0]]) * 255
        
        # Находим и рисуем контуры слоя
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(base_color, contours, -1, color_bgr, 2)
        
    return base_color

def create_gif(save_path, generated_list, original_img_np, distances):
    """
    Создает GIF анимацию (Original + Generated side-by-side).
    """
    frames = []
    original_uint8 = (original_img_np * 255).astype(np.uint8)
    
    for i, gen_img in enumerate(generated_list):
        gen_uint8 = (gen_img * 255).astype(np.uint8)
        
        H, W = original_uint8.shape
        canvas = np.zeros((H, W*2), dtype=np.uint8)
        canvas[:, :W] = original_uint8
        canvas[:, W:] = gen_uint8
        
        # Добавляем текст (B/W -> RGB для цветного текста)
        canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
        text = f"Dist: {distances[i]:.2f}"
        cv2.putText(canvas_rgb, text, (W + 20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        
        frames.append(canvas_rgb)
        
    imageio.mimsave(save_path, frames, fps=2, loop=0)

#ОСНОВНАЯ ФУНКЦИЯ ЗАПУСКА
def run_inference():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    
    # --- Инициализация модели ---
    generator = ConditionalUNet().to(DEVICE)
    generator.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    generator.eval()
    
    print(f"Модель успешно загружена из: {MODEL_PATH}")

    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Найдено изображений для обработки: {len(files)}")

    for fname in files:
        img_path = os.path.join(INPUT_FOLDER, fname)
        # Имя файла без расширения (используем как префикс)
        img_name = os.path.splitext(fname)[0]
        
        # Папка результата для конкретного файла
        save_dir = os.path.join(OUTPUT_ROOT, img_name)
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"Обработка: {fname} ...")
        
        # Загрузка
        raw_pil = Image.open(img_path).convert('L')
        raw_tensor = transform(raw_pil).unsqueeze(0).to(DEVICE)
        
        results_list = []
        
        with torch.no_grad():
            for dist_val in TARGET_DISTANCES:
                norm_dist = dist_val / MAX_DIST
                dist_t = torch.tensor([[norm_dist]], dtype=torch.float32).to(DEVICE)
                
                out = generator(raw_tensor, dist_t)
                out_np = (out.squeeze().cpu().numpy() + 1) / 2
                results_list.append(out_np)
                
                # Сохранение версии: имя_дистанция.png
                version_name = f"{img_name}_{dist_val:.2f}.png"
                Image.fromarray((out_np * 255).astype(np.uint8)).save(os.path.join(save_dir, version_name))

        # Сохранение оригинала: имя.png
        original_name = f"{img_name}.png"
        raw_resized = raw_pil.resize((512, 512))
        raw_resized.save(os.path.join(save_dir, original_name))
        
        raw_np = np.array(raw_resized) / 255.0
        
        # Сохранение оверлея: имя_overlay.png
        overlay = create_overlay_image(results_list, raw_np)
        cv2.imwrite(os.path.join(save_dir, f"{img_name}_overlay.png"), overlay)
        
        # Сохранение GIF: имя_animation.gif
        gif_path = os.path.join(save_dir, f"{img_name}_animation.gif")
        create_gif(gif_path, results_list, raw_np, TARGET_DISTANCES)
        
    print("Все файлы успешно обработаны.")

if __name__ == "__main__":
    run_inference()


FileNotFoundError: [Errno 2] No such file or directory: 'Gen_ep100.pth'